In [ ]:
%load_ext autoreload
%autoreload 2

# C1 — Quality Check

Validates a downloaded camera trap dataset subset.
- Scans every image for corrupt files
- Prints split / class count table
- Reports class balance (max/min ratio)
- Displays a 4×4 random sample grid per class for visual inspection

Set `DATASET` below to `"kgalagadi"` or `"cct20"`.

## Parameters

In [ ]:
from pathlib import Path

DATASET = "cct20"  # "kgalagadi" or "cct20"
REPO_ROOT = Path("../..")  # adjust if running from a different location
DATA_DIR = REPO_ROOT / "data" / DATASET
SPLITS = ["train", "val", "test"]
GRID_COLS = 4
GRID_ROWS = 4  # images per class in the visual grid

assert DATA_DIR.exists(), f"Dataset directory not found: {DATA_DIR.resolve()}"
print(f"Dataset: {DATASET}")
print(f"Path:    {DATA_DIR.resolve()}")

## Imports

In [ ]:
import random
from collections import defaultdict

import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError

## Section 1 — Inventory

Walk the directory tree and build a full index of images by split and class.

In [ ]:
# image_index[split][label] = [path, ...]
image_index = defaultdict(lambda: defaultdict(list))

for split_dir in sorted(DATA_DIR.iterdir()):
    if not split_dir.is_dir() or split_dir.name not in SPLITS:
        continue
    for label_dir in sorted(split_dir.iterdir()):
        if not label_dir.is_dir():
            continue
        for img_path in sorted(label_dir.iterdir()):
            if img_path.suffix.lower() in (".jpg", ".jpeg", ".png"):
                image_index[split_dir.name][label_dir.name].append(img_path)

all_labels = sorted({label for split in image_index.values() for label in split})
print(f"Classes ({len(all_labels)}): {all_labels}")
print(f"Splits found: {sorted(image_index.keys())}")

## Section 2 — Split / Class Count Table

In [ ]:
col_w = 10
header = (
    f"{'label':<25}" + "".join(f"{s:>{col_w}}" for s in SPLITS) + f"{'total':>{col_w}}"
)
print(header)
print("-" * len(header))

totals = defaultdict(int)
grand_total = 0
for label in all_labels:
    counts = [len(image_index[s].get(label, [])) for s in SPLITS]
    row_total = sum(counts)
    grand_total += row_total
    for s, c in zip(SPLITS, counts, strict=False):
        totals[s] += c
    print(
        f"{label:<25}"
        + "".join(f"{c:>{col_w}}" for c in counts)
        + f"{row_total:>{col_w}}"
    )

print("-" * len(header))
print(
    f"{'TOTAL':<25}"
    + "".join(f"{totals[s]:>{col_w}}" for s in SPLITS)
    + f"{grand_total:>{col_w}}"
)

## Section 3 — Class Balance

Compute max/min ratio across classes using the training split only.

In [ ]:
train_counts = {label: len(image_index["train"].get(label, [])) for label in all_labels}
max_count = max(train_counts.values())
min_count = min(v for v in train_counts.values() if v > 0)
ratio = max_count / min_count

print("Train class counts:")
for label, count in sorted(train_counts.items(), key=lambda x: -x[1]):
    bar = "█" * (count * 30 // max_count)
    print(f"  {label:<25} {count:>6}  {bar}")

print(f"\nMax/min ratio (train): {max_count}/{min_count} = {ratio:.1f}x")
if ratio > 5:
    print("⚠ High imbalance — expected for Kgalagadi (empty class dominates).")
else:
    print("✓ Reasonably balanced.")

## Section 4 — Corrupt File Scan

Attempts to open every image with PIL. Logs any files that cannot be decoded.

In [ ]:
all_paths = [
    p for split in image_index.values() for paths in split.values() for p in paths
]
print(f"Scanning {len(all_paths)} images...")

corrupt = []
for i, path in enumerate(all_paths):
    try:
        with Image.open(path) as im:
            im.verify()
    except (UnidentifiedImageError, Exception) as e:
        corrupt.append((path, str(e)))
    if (i + 1) % 500 == 0:
        print(f"  {i + 1}/{len(all_paths)}")

print(f"\nCorrupt files: {len(corrupt)}")
if corrupt:
    for path, err in corrupt:
        print(f"  {path.relative_to(DATA_DIR)} — {err}")
else:
    print("✓ All images are readable.")

## Section 5 — Visual Grid per Class

Displays a 4×4 random sample from the training split for each class. Use this to spot wrong labels, dark IR images, or extreme exposures.

In [ ]:
random.seed(0)

for label in all_labels:
    train_paths = image_index["train"].get(label, [])
    if not train_paths:
        print(f"No training images for class: {label}")
        continue

    sample = random.sample(train_paths, min(GRID_COLS * GRID_ROWS, len(train_paths)))
    n = len(sample)
    n_cols = GRID_COLS
    n_rows = (n + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
    fig.suptitle(
        f"{DATASET} · class: {label} · n_train={len(train_paths)}", fontsize=13, y=1.01
    )
    axes = axes.flatten() if n_rows * n_cols > 1 else [axes]

    for ax, path in zip(axes, sample, strict=False):
        with Image.open(path) as im:
            ax.imshow(im)
        ax.set_title(path.name[:20], fontsize=7)
        ax.axis("off")

    for ax in axes[len(sample) :]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## Summary

Run the cell below to print a one-line QC verdict.

In [ ]:
issues = []
if corrupt:
    issues.append(f"{len(corrupt)} corrupt file(s)")
if ratio > 10:
    issues.append(f"class imbalance ratio {ratio:.1f}x (>10x)")

if issues:
    print(f"⚠ QC issues for [{DATASET}]: {'; '.join(issues)}")
else:
    print(
        f"✓ QC passed for [{DATASET}]: {len(all_paths)} images, {len(all_labels)} classes, {ratio:.1f}x imbalance, 0 corrupt."
    )